In [ ]:
# 필요한 라이브러리 호출
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import numpy as np

# 랜덤함수 호출
rng = np.random.default_rng(0)

In [ ]:
# 평균 0, 표준편차 1인 정규분포에서 크기의 핵심 인자 2개를 1,000개 샘플로 생성
Z = rng.normal(0, 1, (1000, 2)) 
W = rng.normal(0, 1, (2, 6)) 
"""
[1.2, -0.5] @[0.8 -0.4 1.5 0.2 -1.0 0.6] = [1.110, -1.030, 1.550, 0.590, -1.650, 0.520]
    ...      [-0.3 1.1 0.5 -0.7 0.9 0.4]
"""
# Z(1000,2) 와 W(2,6) 행렬곱 에 작은 노이즈를 섞는다. -> Z 와 W 가 얽힘
Xn = Z @ W + rng.normal(0, 0.1, (1000, 6))  #(1000,6)
# 같은 방식으로 (20, 2) @ W + 노이즈를 통해 20개의 샘플을 만든다.
Xa = rng.normal(0, 1, (20, 2)) @ W + rng.normal(0, 0.1, (20, 6)) # (20,6)

# 번째 변수(0번 열)의 부호를 반대로 뒤집어 다른 변수들과의 정상적인 상관관계가 깨진 이상 데이터 20건(Xa)을 만든다.
Xa[:, 0] = -Xa[:, 0]  

[[-0.27412288 -1.2032055  -0.60323248 -2.26440346  0.13034157  3.0063525 ]
 [-0.00982271  1.13240427  0.93860688  3.34496961 -0.01651373 -4.66436022]
 [ 0.40731396  0.24807954 -0.36140539 -1.12301905 -0.16592405  1.78358428]
 ...
 [ 0.83272144  1.47810324  0.05423434  0.53388375 -0.35671594 -0.29512447]
 [-0.40392578 -1.37579455 -0.56446054 -2.17912044  0.18490382  2.81765404]
 [-0.04544119 -0.72000035 -0.52520521 -1.89252593  0.03098752  2.61154662]]


In [8]:
# 쉽게 이야기 해서 Xn 은 정상 데이터, Xa 는 이상데이터 임  

#정상 데이터(Xn) 를 정규화 함
sc6 = StandardScaler().fit(Xn)
# MLP 로 4-2-4 구조를 만든다.
ae = MLPRegressor(hidden_layer_sizes=(4, 2, 4), max_iter=3000, random_state=0)
# 입력과 정답 모두 표준화된 정상 데이터를 넣어서, 입력을 스스로 복원하도록 학습시킴
ae.fit(sc6.transform(Xn), sc6.transform(Xn))

MLPRegressor(hidden_layer_sizes=(4, 2, 4), max_iter=3000, random_state=0)

In [ ]:
def err(D):
    data = sc6.transform(D) # 정규화
    # 원본과 복원본의 차이를 제곱한 후 row 단위로 통계를 내면
    # 오차율을 구할 수 있다. 
    mse = ((data - ae.predict(data)) ** 2).mean(axis=1)
    # 이 역시도 압축 후 복원을 통해 오차를 발견 하므로 재구성 오차이다.
    return mse

In [10]:
print(f'정상 평균 오차 {err(Xn).mean():.3f} / 이상 평균 오차 {err(Xa).mean():.3f}')

정상 평균 오차 0.009 / 이상 평균 오차 24.419


##### 1-4. 임계값과 오탐·미탐 비용

In [20]:
scores = np.r_[err(Xn), err(Xa)] # 점수 배열 [정상 점수...,이상 점수...]
labels = np.r_[np.zeros(len(Xn)), np.ones(len(Xa))] # 정답 라벨 배열 [0(정답)...,1(오답)...]

per = [95, 99, 99.9]
# 오차 분포에서 상위 5%, 1%, 0.1% 에 해당하는 임계값 후보 thr로
for i, thr in enumerate(np.percentile(err(Xn), per)):
    print(f'오차분포 상위 {100-per[i]:.1f}% 의 임계값 : {thr:.3f}')
    # False Positive(오탐) = 점수배열 중 임계값을 넘겨 오차판정 했으나 정답(labels == 0)인 갯수
    fp = ((scores > thr) & (labels == 0)).sum()
    # False Negative(미탐) = 점수배열 중 임계값을 넘지않아 정상판정 했으나 오답(labels == 1)인 갯수
    fn = ((scores <= thr) & (labels == 1)).sum()
    # 현장에서 오탐의 가중치:1, 미탐 가중치: 50 라고 했다 가정해 보자
    print(f'    # 오탐:{fp} / 미탐:{fn} / 비용:{fp * 1 + fn * 50}')

오차분포 상위 5.0% 의 임계값 : 0.018
    # 오탐:50 / 미탐:0 / 비용:50
오차분포 상위 1.0% 의 임계값 : 0.239
    # 오탐:10 / 미탐:0 / 비용:10
오차분포 상위 0.1% 의 임계값 : 0.633
    # 오탐:1 / 미탐:0 / 비용:1
